# Bối cảnh, câu chuyện dữ liệu và giả thuyết giá–điểm

**Người phụ trách:** Dương. Bản hiệu chỉnh cục bộ để tích hợp báo cáo tiến trình.

**Mục tiêu:** giải thích vì sao nhóm phân tích giá–điểm, xác định mẫu đang dùng,
diễn giải quan sát và nối sang câu hỏi dự đoán.

**Trạng thái:** EDA sơ bộ trên mẫu có giá; chưa có kết quả Spearman, mô hình,
validation hoặc test. Split seed 42 đã chốt ngày 21/10/2026; kiểm định dành cho buổi sau.

## 1. Bối cảnh và nguồn dữ liệu

Giá thường được xem như một tín hiệu về chất lượng. Nhóm đặt câu hỏi liệu các
bản ghi có giá cao hơn có xu hướng được chấm điểm cao hơn hay không, thay vì
mặc định rằng giá cao luôn đi cùng điểm cao.

Nguồn gốc là Wine Reviews 130k, được lưu tại
[CSV raw](../../data/raw/winemag-data-130k-v2.csv). Tên “130k” nói về nguồn gốc,
không phải số bản ghi sau làm sạch. Các số liệu trong notebook này tính trực tiếp từ
[wine_cleaned_final.csv](../../data/processed/wine_cleaned_final.csv);
không lấy thống kê raw trong audit làm số liệu của mẫu sạch.

Mỗi dòng là **một bản ghi review**, chưa chắc là một chai duy nhất.
`points` là điểm do người review chấm; không phải phép đo khách quan và đầy đủ
về chất lượng rượu hoặc sở thích của mọi người mua.

## 2. Mẫu phân tích và thống kê có thể chạy lại

Chỉ đọc dữ liệu sạch một lần. `df_all` giữ toàn bộ bản ghi sạch; `df_price`
là bản sao các bản ghi có giá. Không điền giá thiếu, chạy lại cleaning hoặc
đọc CSV split. Mở từ project root, `notebooks` hoặc `w5_progress_parts`,
chọn kernel môi trường dự án rồi Restart Kernel & Run All.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "w5_progress_parts":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wine_cleaned_final.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError("Chưa có dữ liệu sạch. Mở notebook từ project root hoặc thư mục notebook.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Chỉ đọc CSV một lần. Không chạy cleaning và không đọc các file split.
df_all = pd.read_csv(DATA_PATH)
required_columns = {
    "price", "points", "log_price", "title", "country", "province",
    "variety", "vintage_year", "price_was_missing",
}
missing_columns = required_columns - set(df_all.columns)
if missing_columns:
    raise ValueError(f"Dữ liệu sạch thiếu cột: {sorted(missing_columns)}")
if df_all.empty:
    raise ValueError("Dữ liệu sạch không có bản ghi.")
if not df_all["price"].isna().equals(df_all["price_was_missing"]):
    raise ValueError("Cờ price_was_missing không khớp trạng thái thiếu giá.")

# df_all luôn là toàn bộ dữ liệu sạch; df_price là bản sao có giá quan sát.
df_price = df_all.loc[df_all["price"].notna()].copy()
if df_price.empty:
    raise ValueError("Không có bản ghi có giá để phân tích.")
for column in ["price", "points"]:
    df_price[column] = pd.to_numeric(df_price[column], errors="raise")
if not (np.isfinite(df_price["price"]) & df_price["price"].gt(0)).all():
    raise ValueError("Giá quan sát phải dương và hữu hạn.")
if not (np.isfinite(df_price["points"]) & df_price["points"].between(80, 100)).all():
    raise ValueError("Điểm review phải đầy đủ, hữu hạn và trong khoảng 80–100.")
if not np.allclose(df_price["log_price"], np.log1p(df_price["price"])):
    raise ValueError("log_price không khớp log1p(price).")

def format_number(value, decimals=0):
    return f"{value:,.{decimals}f}".translate(str.maketrans({",": ".", ".": ","}))

def format_price(value):
    return format_number(value, 2).rstrip("0").rstrip(",")

n_total = len(df_all)
n_with_price = len(df_price)
n_missing_price = int(df_all["price"].isna().sum())
assert n_with_price + n_missing_price == n_total
print(f"Dữ liệu sạch: {format_number(n_total)} bản ghi, {len(df_all.columns)} cột.")
print(f"Mẫu có giá: {format_number(n_with_price)}; thiếu giá: {format_number(n_missing_price)}.")
print("Trạng thái: EDA sơ bộ toàn mẫu có giá; split seed 42 đã chốt; chưa có kiểm định/mô hình.")

Dữ liệu sạch: 119.988 bản ghi, 19 cột.
Mẫu có giá: 111.593; thiếu giá: 8.395.
Trạng thái: EDA sơ bộ toàn mẫu có giá; split seed 42 đã chốt; chưa có kiểm định/mô hình.


In [2]:
sample_summary = pd.DataFrame({
    "Nhóm bản ghi": [
        "Toàn bộ dữ liệu sạch",
        "Có nhãn points",
        "Có giá quan sát — mẫu mô tả",
        "Thiếu giá — giữ lại trong dữ liệu sạch",
    ],
    "Số bản ghi": [
        n_total, int(df_all["points"].notna().sum()), n_with_price, n_missing_price,
    ],
})
sample_summary["Tỷ lệ trên dữ liệu sạch (%)"] = (
    sample_summary["Số bản ghi"] / n_total * 100
).round(2)
sample_summary

,Nhóm bản ghi,Số bản ghi,Tỷ lệ trên dữ liệu sạch (%)
0,Toàn bộ dữ liệu sạch,119988,100.0
1,Có nhãn points,119988,100.0
2,Có giá quan sát — mẫu mô tả,111593,93.0
3,Thiếu giá — giữ lại trong dữ liệu sạch,8395,7.0


### Phân bố giá của mẫu có giá

Các mốc dưới đây tính trên `df_price`. Tứ phân vị là mốc 25%, 50% và 75%;
mốc 50% là trung vị. Trung vị ít bị kéo bởi một số mức giá rất cao hơn trung bình.
Đây là bảng mô tả nguồn cho lời dẫn, không tạo lại bảng bốn nhóm giá của Khánh.

In [3]:
price_stats = df_price["price"].describe(percentiles=[0.25, 0.5, 0.75])
price_summary = pd.DataFrame({
    "Thống kê giá": ["Nhỏ nhất", "Mốc 25%", "Trung vị (50%)", "Mốc 75%", "Trung bình", "Lớn nhất"],
    "Giá (USD)": price_stats.loc[["min", "25%", "50%", "75%", "mean", "max"]].to_numpy(),
})
price_summary.round(2)

,Thống kê giá,Giá (USD)
0,Nhỏ nhất,4.00
1,Mốc 25%,17.00
2,Trung vị (50%),25.00
3,Mốc 75%,42.00
4,Trung bình,35.57
5,Lớn nhất,2500.00


In [4]:
n_above_300 = int(df_price["price"].gt(300).sum())
points_min = int(df_all["points"].min())
points_max = int(df_all["points"].max())
display(Markdown(
    f"**Phạm vi thực tế:** {format_number(n_total)} bản ghi sạch; "
    f"{format_number(n_with_price)} có giá "
    f"({format_number(n_with_price / n_total * 100, 2)}%); "
    f"{format_number(n_missing_price)} thiếu giá "
    f"({format_number(n_missing_price / n_total * 100, 2)}%). "
    f"Điểm review trong dữ liệu sạch nằm từ {points_min} đến {points_max}. "
    f"Giá trung vị {format_price(price_stats['50%'])} USD; "
    f"giá cao nhất {format_price(price_stats['max'])} USD. "
    f"Có {format_number(n_above_300)} bản ghi giá trên 300 USD; "
    "chúng vẫn nằm trong thống kê dù nằm ngoài khung hình 0–300 USD của Đô."
))

**Phạm vi thực tế:** 119.988 bản ghi sạch; 111.593 có giá (93,00%); 8.395 thiếu giá (7,00%). Điểm review trong dữ liệu sạch nằm từ 80 đến 100. Giá trung vị 25 USD; giá cao nhất 2.500 USD. Có 310 bản ghi giá trên 300 USD; chúng vẫn nằm trong thống kê dù nằm ngoài khung hình 0–300 USD của Đô.

## 3. Vấn đề và câu hỏi nghiên cứu

**Câu hỏi khám phá:** Trong mẫu có giá, phân bố giá ra sao? Điểm review thay đổi
thế nào giữa bốn khoảng giá? Cùng mức giá có thể có nhiều mức điểm không?

**Câu hỏi dự án:** Chỉ biết giá, dự đoán `points` tốt hơn mức dự đoán cơ bản
(Dummy) bao nhiêu? Thêm `country`, `province`, `variety` và `vintage_year` có giảm
sai số so với chỉ dùng giá không?

Bảng bốn khoảng giá của Khánh và hai hình của Đô cung cấp bằng chứng mô tả ban đầu.
Việc so sánh sai số trên cùng mẫu và cùng split ở giai đoạn sau mới đánh giá
khả năng dự đoán và giá trị bổ sung của metadata.

## 4. Giả thuyết và trạng thái kiểm định

**Giả thuyết về mối liên hệ:** bản ghi giá cao hơn có xu hướng nhận điểm review
cao hơn. Đây là kỳ vọng về xu hướng chung, không phải quy luật đúng cho từng review.

**Giả thuyết dự đoán:** metadata có thể bổ sung thông tin ngoài giá. Đây là điều
cần kiểm chứng; chưa được coi là kết quả từ các bảng và hình mô tả.

**Chưa có kết quả Spearman.** Split seed 42 đã chốt; thực hiện Spearman sơ bộ trên train ở buổi sau, báo số mẫu, hệ số, p-value và giới hạn. Không thực hiện
trên toàn bộ mẫu để công bố kiểm định trong báo cáo EDA này. Kết quả sơ bộ
không thay thế kiểm định chính thức theo roadmap.

## 5. Câu chuyện từ những gì đã quan sát

Đọc cùng [bảng bốn khoảng giá của Khánh](01_khanh_price_tiers.ipynb) và
[hai hình của Đô](03_do_eda_plots.ipynb). Phần dưới đối chiếu trực tiếp trên
mẫu có giá; không tạo thêm biểu đồ hoặc huấn luyện mô hình.

In [5]:
points_at_25 = df_price.loc[df_price["price"].eq(25), "points"]
if points_at_25.empty:
    raise ValueError("Không có bản ghi giá 25 USD để đối chiếu nhận xét.")
same_price_n = len(points_at_25)
same_price_min = int(points_at_25.min())
same_price_max = int(points_at_25.max())
display(Markdown(
    f"**Quan sát 1 — Phân bố giá:** trung vị là {format_price(price_stats['50%'])} USD, "
    f"trung bình là {format_number(price_stats['mean'], 1)} USD và "
    f"giá cao nhất {format_price(price_stats['max'])} USD. "
    "Các mốc này và histogram của Đô giúp nhận diện phần đuôi giá cao.\n\n"
    f"**Quan sát 2 — Cùng giá, khác điểm:** tại mức 25 USD có "
    f"{format_number(same_price_n)} bản ghi, với điểm từ {same_price_min} đến {same_price_max}. "
    "Vì vậy, chỉ biết giá chưa xác định được điểm của từng bản ghi.\n\n"
    "**Câu hỏi còn mở:** metadata có giúp dự đoán sát hơn không? "
    "Cần so sánh mô hình trên cùng validation trên split seed 42 đã chốt; "
    "các quan sát này chưa chứng minh lợi ích dự đoán bổ sung."
))

**Quan sát 1 — Phân bố giá:** trung vị là 25 USD, trung bình là 35,6 USD và giá cao nhất 2.500 USD. Các mốc này và histogram của Đô giúp nhận diện phần đuôi giá cao.

**Quan sát 2 — Cùng giá, khác điểm:** tại mức 25 USD có 5.320 bản ghi, với điểm từ 80 đến 95. Vì vậy, chỉ biết giá chưa xác định được điểm của từng bản ghi.

**Câu hỏi còn mở:** metadata có giúp dự đoán sát hơn không? Cần so sánh mô hình trên cùng validation trên split seed 42 đã chốt; các quan sát này chưa chứng minh lợi ích dự đoán bổ sung.

## 6. Giới hạn diễn giải

- Mẫu review được công bố không mặc nhiên đại diện toàn bộ rượu trên thị trường;
  điểm trong dữ liệu chỉ nằm trong khoảng quan sát đã nêu.
- Chỉ dùng bản ghi có giá có thể tạo thiên lệch nếu nhóm này khác nhóm thiếu giá.
  Số thiếu giá của mẫu sạch được tính ở bảng trên, không dùng số thiếu giá raw.
- `points` phản ánh đánh giá của người chấm, có thể khác cảm nhận người tiêu dùng.
- Giá và điểm có thể cùng liên quan đến quốc gia, vùng, giống nho, niên vụ hoặc
  người đánh giá. Tương quan không chứng minh giá cao gây ra điểm cao.
- Giá có thể phụ thuộc thời điểm, thị trường và cách thu thập.
- Bốn khoảng giá là nhóm tương đối trong mẫu, không phải chuẩn phân khúc cố định.
- Chưa có phép kiểm định hay đánh giá trên dữ liệu chưa thấy; không suy ra mô hình
  dự đoán tốt chỉ từ xu hướng trên toàn bộ mẫu có giá.

## 7. Bước tiếp theo và tài liệu

Khánh tích hợp nội dung đã kiểm tra vào [notebook chung](../02_eda.ipynb).
Buổi sau, trên split seed 42 đã chốt: kiểm tra Spearman sơ bộ trên train; nhóm chạy baseline
và so sánh cấu hình trên cùng train/validation theo roadmap.
Test chỉ dùng ở bước đánh giá cuối sau khi giao thức và mô hình được khóa.

- [Roadmap](../../docs/ROADMAP_W5_W10.md): phạm vi từng tuần.
- [Data dictionary](../../docs/data_dictionary.md): định nghĩa cột.
- [Data audit](../../docs/data_audit.md): audit nguồn, không thay thế thống kê mẫu sạch.
- [README](../../README.md): câu hỏi dự án và cấu hình mô hình dự kiến.